<a href="https://colab.research.google.com/github/1nazrinaliyeva/GLiNER_train/blob/main/GLINER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U gliner datasets accelerate seqeval sentencepiece


In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("LocalDoc/azerbaijani-ner-dataset")
print(ds)
print(ds["train"][0])

In [ ]:
import ast
from datasets import load_dataset, Dataset, DatasetDict

raw_ds = load_dataset("LocalDoc/azerbaijani-ner-dataset")

def safe_parse_list(x):
    if x is None:
        return None
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        return ast.literal_eval(x)
    return None

def normalize_tokens(x):
    x = safe_parse_list(x)
    if x is None:
        return None
    return [str(t) for t in x]

def normalize_tags(x):
    x = safe_parse_list(x)
    if x is None:
        return None
    try:
        return [int(t) for t in x]
    except Exception:
        return None

def convert_split(split, split_name):
    rows = []
    skipped = 0

    for i, ex in enumerate(split):
        tokens = normalize_tokens(ex.get("tokens"))
        ner_tags = normalize_tags(ex.get("ner_tags"))

        if tokens is None or ner_tags is None:
            skipped += 1
            continue

        if len(tokens) != len(ner_tags):
            skipped += 1
            continue

        rows.append({
            "index": str(ex.get("index")),
            "tokens": tokens,
            "ner_tags": ner_tags,
        })

    print(split_name, "kechilen/buraxilan setirler:", skipped)
    return Dataset.from_list(rows)

parts = {}
for split_name in raw_ds.keys():
    parts[split_name] = convert_split(raw_ds[split_name], split_name)

ds = DatasetDict(parts)

print(ds)
print(ds["train"][0])
print(type(ds["train"][0]["tokens"]))
print(type(ds["train"][0]["ner_tags"]))
print(type(ds["train"][0]["ner_tags"][0]))


In [ ]:
split1 = ds["train"].train_test_split(test_size=0.1, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

final_ds = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"],
})

print(final_ds)


In [ ]:
import json

id2label = {
    0: "O",
    1: "PERSON",
    2: "LOCATION",
    3: "ORGANISATION",
    4: "DATE",
    5: "TIME",
    6: "MONEY",
    7: "PERCENTAGE",
    8: "FACILITY",
    9: "PRODUCT",
    10: "EVENT",
    11: "ART",
    12: "LAW",
    13: "LANGUAGE",
    14: "GPE",
    15: "NORP",
    16: "ORDINAL",
    17: "CARDINAL",
    18: "DISEASE",
    19: "CONTACT",
    20: "ADAGE",
    21: "QUANTITY",
    22: "MISCELLANEOUS",
    23: "POSITION",
    24: "PROJECT",
}

def to_gliner_example(example):
    tokens = example["tokens"]
    tags = example["ner_tags"]

    ner = []
    current_label = None
    start_idx = None

    for i, tag_id in enumerate(tags):
        if tag_id == 0:
            if current_label is not None:
                ner.append([start_idx, i - 1, current_label])
                current_label = None
                start_idx = None
            continue

        label = id2label[tag_id].lower()

        if current_label is None:
            current_label = label
            start_idx = i
        elif label != current_label:
            ner.append([start_idx, i - 1, current_label])
            current_label = label
            start_idx = i

    if current_label is not None:
        ner.append([start_idx, len(tags) - 1, current_label])

    return {
        "tokenized_text": tokens,
        "ner": ner,
    }

def convert_split(split, drop_empty=True):
    rows = []
    for ex in split:
        row = to_gliner_example(ex)
        if drop_empty and len(row["ner"]) == 0:
            continue
        rows.append(row)
    return rows

train_data = convert_split(final_ds["train"], drop_empty=True)
val_data = convert_split(final_ds["validation"], drop_empty=True)
test_data = convert_split(final_ds["test"], drop_empty=False)

print(len(train_data), len(val_data), len(test_data))
print(train_data[0])

with open("train_gliner.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False)

with open("val_gliner.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False)

with open("test_gliner.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False)


In [ ]:
import gc
import torch
from gliner import GLiNER

gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

MAX_TOKENS = 96

train_data_small = [
    x for x in train_data
    if 1 <= len(x["tokenized_text"]) <= MAX_TOKENS
][:10000]

val_data_small = [
    x for x in val_data
    if 1 <= len(x["tokenized_text"]) <= MAX_TOKENS
][:1000]

print(len(train_data_small), len(val_data_small))
print(max(len(x["tokenized_text"]) for x in train_data_small))


In [ ]:
model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

model.train_model(
    train_dataset=train_data_small,
    eval_dataset=val_data_small,
    output_dir="gliner_az_model",
    max_steps=300,
    learning_rate=2e-5,
    others_lr=1e-5,
    weight_decay=0.01,
    others_weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    logging_steps=20,
    save_steps=150,
    save_total_limit=1,
)


## Preprocessing



In [ ]:
id2label = {
    0: "O",
    1: "PERSON",
    2: "LOCATION",
    3: "ORGANISATION",
    4: "DATE",
    5: "TIME",
    6: "MONEY",
    7: "PERCENTAGE",
    8: "FACILITY",
    9: "PRODUCT",
    10: "EVENT",
    11: "ART",
    12: "LAW",
    13: "LANGUAGE",
    14: "GPE",
    15: "NORP",
    16: "ORDINAL",
    17: "CARDINAL",
    18: "DISEASE",
    19: "CONTACT",
    20: "ADAGE",
    21: "QUANTITY",
    22: "MISCELLANEOUS",
    23: "POSITION",
    24: "PROJECT",
}


In [ ]:
from collections import Counter

label_token_counts = Counter()
label_row_counts = Counter()
broken_rows = 0

for ex in final_ds["train"]:
    tokens = ex["tokens"]
    tags = ex["ner_tags"]

    if tokens is None or tags is None or len(tokens) != len(tags):
        broken_rows += 1
        continue

    labels_in_this_row = set()

    for tag in tags:
        if tag != 0:
            label = id2label[tag]
            label_token_counts[label] += 1
            labels_in_this_row.add(label)

    for label in labels_in_this_row:
        label_row_counts[label] += 1

print("broken row sayi:", broken_rows)
print("\nToken sayina gore label statistikasi:")
for label, count in label_token_counts.most_common():
    print(label, count)

print("\nRow sayina gore label statistikasi:")
for label, count in label_row_counts.most_common():
    print(label, count)


In [ ]:
import random

def show_example(ex):
    print("TEXT:", " ".join(ex["tokens"]))
    print("ENTITIES:")
    for token, tag in zip(ex["tokens"], ex["ner_tags"]):
        if tag != 0:
            print(f"  {token} -> {id2label[tag]}")
    print("-" * 50)

sample_ids = random.sample(range(len(final_ds["train"])), 100)

for idx in sample_ids:
    show_example(final_ds["train"][idx])


## Bu nümunələrdən görünür ki, datasetdə bəzi label-lar, xüsusilə `MISCELLANEOUS` və `ORGANISATION`, həddindən artıq geniş və bəzən sözün mənasına uyğun olmayan şəkildə istifadə olunub. Həmçinin söz birləşməsi şəklində olan xüsusi adlar çox vaxt bütöv şəkildə deyil, ayrı-ayrı tokenlər kimi etiketlənib. Bu da datasetdə etiketləmə qaydasının hər yerdə eyni şəkildə tətbiq olunmadığını və annotasiya keyfiyyətində problemlər olduğunu göstərir. Bundan əlavə, label sayının çox olması bir-birinə yaxın kateqoriyalar arasında qarışıqlıq yaradır və modelin öyrənməsini çətinləşdirə bilər.


In [ ]:
id2label = {
    0: "O",
    1: "PERSON",
    2: "LOCATION",
    3: "ORGANISATION",
    4: "DATE",
    5: "TIME",
    6: "MONEY",
    7: "PERCENTAGE",
    8: "FACILITY",
    9: "PRODUCT",
    10: "EVENT",
    11: "ART",
    12: "LAW",
    13: "LANGUAGE",
    14: "GPE",
    15: "NORP",
    16: "ORDINAL",
    17: "CARDINAL",
    18: "DISEASE",
    19: "CONTACT",
    20: "ADAGE",
    21: "QUANTITY",
    22: "MISCELLANEOUS",
    23: "POSITION",
    24: "PROJECT",
}

keep_labels = {
    "PERSON",
    "LOCATION",
    "ORGANISATION",
    "DATE",
    "TIME",
    "MONEY",
    "PRODUCT",
    "EVENT",
    "CONTACT",
    "FACILITY",
    "POSITION",
}

def simplify_label(label):
    if label == "GPE":
        return "LOCATION"
    if label in keep_labels:
        return label
    return "O"


In [ ]:
def convert_to_gliner(split):
    rows = []

    for ex in split:
        tokens = ex["tokens"]
        tags = ex["ner_tags"]

        ner = []
        current_label = None
        start_idx = None

        for i, tag_id in enumerate(tags):
            raw_label = id2label[tag_id]
            label = simplify_label(raw_label)

            if label == "O":
                if current_label is not None:
                    ner.append([start_idx, i - 1, current_label.lower()])
                    current_label = None
                    start_idx = None
                continue

            if current_label is None:
                current_label = label
                start_idx = i
            elif current_label != label:
                ner.append([start_idx, i - 1, current_label.lower()])
                current_label = label
                start_idx = i

        if current_label is not None:
            ner.append([start_idx, len(tags) - 1, current_label.lower()])

        if len(ner) > 0:
            rows.append({
                "tokenized_text": tokens,
                "ner": ner,
            })

    return rows

train_simple = convert_to_gliner(final_ds["train"])
val_simple = convert_to_gliner(final_ds["validation"])
test_simple = convert_to_gliner(final_ds["test"])

print(len(train_simple), len(val_simple), len(test_simple))
print(train_simple[0])


In [ ]:
import random

def show_example(ex):
    print("TEXT:", " ".join(ex["tokens"]))
    print("ENTITIES:")
    for token, tag in zip(ex["tokens"], ex["ner_tags"]):
        if tag != 0:
            print(f"  {token} -> {id2label[tag]}")
    print("-" * 50)

sample_ids = random.sample(range(len(final_ds["train"])), 10)

for idx in sample_ids:
    show_example(final_ds["train"][idx])


In [ ]:
for i in range(3):
    print("TEXT:", " ".join(train_simple[i]["tokenized_text"]))
    print("NER:", train_simple[i]["ner"])
    print("-" * 50)


In [ ]:
train_tiny = [x for x in train_simple if len(x["tokenized_text"]) <= 32][:1000]
val_tiny = [x for x in val_simple if len(x["tokenized_text"]) <= 32][:100]

print(len(train_tiny), len(val_tiny))
print(max(len(x["tokenized_text"]) for x in train_tiny))


In [ ]:
import os
import gc
import torch
from gliner import GLiNER

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

train_tiny2 = [x for x in train_simple if len(x["tokenized_text"]) <= 24][:300]
val_tiny2 = [x for x in val_simple if len(x["tokenized_text"]) <= 24][:30]

print(len(train_tiny2), len(val_tiny2))

model = GLiNER.from_pretrained("urchade/gliner_small-v2.1")

trainer = model.train_model(
    train_dataset=train_tiny2,
    eval_dataset=val_tiny2,
    output_dir="gliner_az_model_small_v2",
    max_steps=20,
    learning_rate=2e-5,
    others_lr=1e-5,
    weight_decay=0.01,
    others_weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    logging_steps=5,
    save_steps=100,
    save_total_limit=1,
)
